# AI CAPEX and GDP — Simplified v3

This notebook implements the framework in `Framework_gdp_ai_capex.docx` while preserving the linear, auditable structure of v2.

The central sequence is:

1. Estimate gross nominal US AI investment from company CAPEX.
2. Apply asset-specific GDP-recognition timing.
3. Estimate and subtract imports **in nominal dollars**.
4. Build independent chained price indexes for gross and net AI investment.
5. Deflate the gross and net aggregates independently.
6. Estimate approximate gross and net contributions to real GDP growth.

The model never defines real net investment as real gross investment minus real imports. All real contribution estimates are approximate because custom chained-dollar flows are not strictly additive to official real GDP.


In [1]:
from pathlib import Path
import json
import math
import subprocess
import urllib.request
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)

# -----------------------------------------------------------------------------
# Files, dates, and execution controls
# -----------------------------------------------------------------------------
ROOT = Path.cwd()
INPUT_FILE = "capex_final_data.csv"
OUTPUT_FILE = "results_simplified_v3.xlsx"
CSV_PATH = ROOT / INPUT_FILE
OUTPUT_DIR = ROOT / "output"
CACHE_DIR = ROOT / "data_cache"
WORKBOOK_BUILDER = ROOT / "workbook_build_v3" / "build_v3_workbook.mjs"
BUNDLED_NODE = Path(r"C:\Users\abour\.cache\codex-runtimes\codex-primary-runtime\dependencies\node\bin\node.exe")

LAST_ACTUAL_QUARTER = pd.Period("2026Q2", freq="Q")
MODEL_START = pd.Period("2022Q1", freq="Q")
ANALYSIS_END = pd.Period("2028Q4", freq="Q")
REAL_CONTRIBUTION_END = LAST_ACTUAL_QUARTER
BASELINE_YEAR = 2022
QOQ_ANNUALIZATION_EXPONENT = 4
IMPORT_SHARE_LOOKBACK_QUARTERS = 2
PROFILE_PRICE_DRIFT_QUARTERS = 12
REFRESH_BEA = False
EXPORT_OUTPUTS = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# Central attribution and geography assumptions
# -----------------------------------------------------------------------------
BASELINE_HYPERSCALERS = {
    "Amazon.com Inc",
    "Alphabet Inc",
    "Microsoft Corp",
    "Meta Platforms Inc",
    "Oracle Corp",
}
US_SHARE_DEFAULT = 0.70

# Baseline assignment is deliberately separate from the capital profile.
PROFILE_OVERRIDES = {
    "Microsoft Corp": "hyperscaler",
    "Amazon.com Inc": "hyperscaler",
    "Alphabet Inc": "hyperscaler",
    "Meta Platforms Inc": "hyperscaler",
    "Oracle Corp": "hyperscaler",
    "CoreWeave Inc": "hyperscaler",
    "Applied Digital Corp": "hyperscaler",
    "DigitalOcean Holdings Inc": "hyperscaler",
    "Cloudflare Inc": "hyperscaler",
    "Tesla Inc": "tesla_automotive",
}

INDUSTRY_PROFILE = {
    "Software & Services": "software_services",
    "Technology Hardware & Equipment": "tech_hardware",
    "Utilities": "utilities",
    "Media & Entertainment": "hyperscaler",
    "Automobiles & Components": "tesla_automotive",
    "Consumer Discretionary Distribution & Retail": "hyperscaler",
}

CATEGORY_SPLITS = pd.DataFrame.from_dict(
    {
        "hyperscaler": {
            "dc_structures": 0.25,
            "power_structures": 0.00,
            "it_equipment": 0.65,
            "power_cooling_equipment": 0.10,
        },
        "software_services": {
            "dc_structures": 0.20,
            "power_structures": 0.00,
            "it_equipment": 0.65,
            "power_cooling_equipment": 0.15,
        },
        "tech_hardware": {
            "dc_structures": 0.10,
            "power_structures": 0.00,
            "it_equipment": 0.75,
            "power_cooling_equipment": 0.15,
        },
        "utilities": {
            "dc_structures": 0.00,
            "power_structures": 0.70,
            "it_equipment": 0.00,
            "power_cooling_equipment": 0.30,
        },
        "tesla_automotive": {
            "dc_structures": 0.20,
            "power_structures": 0.25,
            "it_equipment": 0.40,
            "power_cooling_equipment": 0.15,
        },
    },
    orient="index",
)

LAG_WEIGHTS = {
    "dc_structures": [0.30, 0.30, 0.20, 0.20],
    "power_structures": [0.25, 0.25, 0.25, 0.25],
    "it_equipment": [0.60, 0.30, 0.10],
    "power_cooling_equipment": [0.50, 0.30, 0.20],
}

V2_FIXED_IMPORT_SHARES = {
    "dc_structures": 0.00,
    "power_structures": 0.00,
    "it_equipment": 0.60,
    "power_cooling_equipment": 0.30,
}

assert np.allclose(CATEGORY_SPLITS.sum(axis=1), 1.0)
assert all(np.isclose(sum(weights), 1.0) for weights in LAG_WEIGHTS.values())

# -----------------------------------------------------------------------------
# Official BEA bulk-data registry. Semiconductor import codes are intentionally
# absent from both the nominal and price registries.
# -----------------------------------------------------------------------------
BEA_BULK_FILES = {
    "section1": (
        "bea_section1.xlsx",
        "https://apps.bea.gov/national/Release/XLS/Survey/Section1All_xls.xlsx",
    ),
    "section4": (
        "bea_section4.xlsx",
        "https://apps.bea.gov/national/Release/XLS/Survey/Section4All_xls.xlsx",
    ),
    "underlying5": (
        "bea_underlying_section5.xlsx",
        "https://apps.bea.gov/national/Release/XLS/Underlying/Section5All_xls.xlsx",
    ),
}

BEA_SERIES_REGISTRY = {
    "real_gdp": dict(bulk="section1", sheet="T10106-Q", code="A191RX", label="Gross domestic product", scale=0.001, units="Billions of chained 2017 dollars, SAAR"),
    "import_capital_goods": dict(bulk="section4", sheet="T40205B-Q", code="A650RC", label="Capital goods and related parts", scale=1.0, units="$ millions, SAAR"),
    "import_consumer_goods": dict(bulk="section4", sheet="T40205B-Q", code="A652RC", label="Consumer goods, except food", scale=1.0, units="$ millions, SAAR"),
    "computer_imports": dict(bulk="section4", sheet="T40205B-Q", code="B852RC", label="Computers, peripherals, and parts", scale=1.0, units="$ millions, SAAR"),
    "telecom_imports": dict(bulk="section4", sheet="T40205B-Q", code="LA001146", label="Telecommunications equipment", scale=1.0, units="$ millions, SAAR"),
    "electrical_imports": dict(bulk="section4", sheet="T40205B-Q", code="LA001149", label="Electrical equipment and parts", scale=1.0, units="$ millions, SAAR"),
    "computers_nominal": dict(bulk="underlying5", sheet="U50505-Q", code="B935RC", label="Computers and peripheral equipment", scale=1.0, units="$ millions, SAAR"),
    "communication_nominal": dict(bulk="underlying5", sheet="U50505-Q", code="C275RC", label="Communication equipment", scale=1.0, units="$ millions, SAAR"),
    "electrical_nominal": dict(bulk="underlying5", sheet="U50505-Q", code="C276RC", label="Electrical transmission", scale=1.0, units="$ millions, SAAR"),
    "dc_structures_price": dict(bulk="underlying5", sheet="U50404-Q", code="IA001282", label="Data centers", scale=1.0, units="2017=100"),
    "power_structures_price": dict(bulk="underlying5", sheet="U50404-Q", code="W028RG", label="Power", scale=1.0, units="2017=100"),
    "computers_price": dict(bulk="underlying5", sheet="U50504-Q", code="B935RG", label="Computers and peripheral equipment", scale=1.0, units="2017=100"),
    "communication_price": dict(bulk="underlying5", sheet="U50504-Q", code="B275RG", label="Communication equipment", scale=1.0, units="2017=100"),
    "electrical_price": dict(bulk="underlying5", sheet="U50504-Q", code="B276RG", label="Electrical transmission", scale=1.0, units="2017=100"),
}

FORBIDDEN_SEMICONDUCTOR_CODES = {"LA001145", "IA001145"}

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Central sequence: nominal gross -> nominal imports -> nominal net -> chained deflator -> real flow")
display(CATEGORY_SPLITS.map(lambda value: f"{value:.0%}"))


Central sequence: nominal gross -> nominal imports -> nominal net -> chained deflator -> real flow
                  dc_structures power_structures it_equipment power_cooling_equipment
hyperscaler                 25%               0%          65%                     10%
software_services           20%               0%          65%                     15%
tech_hardware               10%               0%          75%                     15%
utilities                    0%              70%           0%                     30%
tesla_automotive            20%              25%          40%                     15%


## 2. Company CAPEX and central AI attribution

Only the five named baseline hyperscalers subtract a same-calendar-quarter 2022 baseline. All other reported CAPEX is included in the central AI measure. Missing observations are never interpolated or replaced with zero.


In [2]:
REQUIRED_COLUMNS = ["Quarter_End_Date", "Company_Name", "GICS_Industry_Group", "CAPEX"]
if not CSV_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {CSV_PATH}")

capex_raw = pd.read_csv(CSV_PATH)
missing_columns = set(REQUIRED_COLUMNS).difference(capex_raw.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

capex_raw = capex_raw[REQUIRED_COLUMNS].copy()
capex_raw["Quarter_End_Date"] = pd.to_datetime(capex_raw["Quarter_End_Date"], errors="raise")
capex_raw["quarter"] = capex_raw["Quarter_End_Date"].dt.to_period("Q")
capex_raw["CAPEX"] = pd.to_numeric(capex_raw["CAPEX"], errors="coerce")
capex_raw["Company_Name"] = capex_raw["Company_Name"].astype(str).str.strip()
capex_raw["GICS_Industry_Group"] = capex_raw["GICS_Industry_Group"].astype(str).str.strip()

duplicate_count = int(capex_raw.duplicated(["Company_Name", "quarter"]).sum())
blank_industry_rows = int(capex_raw["GICS_Industry_Group"].replace("", np.nan).isna().sum())
industry_conflicts = int((capex_raw.groupby("Company_Name")["GICS_Industry_Group"].nunique(dropna=True) != 1).sum())
total_2024 = capex_raw.loc[capex_raw["quarter"].dt.year.eq(2024), "CAPEX"].sum(min_count=1)

input_audit = pd.DataFrame(
    {
        "metric": [
            "rows",
            "companies",
            "duplicate company-quarters",
            "blank industry rows",
            "companies with conflicting industries",
            "missing CAPEX values",
            "negative reported CAPEX values",
            "2024 CAPEX ($bn)",
        ],
        "value": [
            len(capex_raw),
            capex_raw["Company_Name"].nunique(),
            duplicate_count,
            blank_industry_rows,
            industry_conflicts,
            int(capex_raw["CAPEX"].isna().sum()),
            int(capex_raw["CAPEX"].lt(0).sum()),
            total_2024,
        ],
    }
)

if duplicate_count or blank_industry_rows or industry_conflicts:
    raise ValueError("Company-quarter or industry validation failed.")
if abs(total_2024 - 490.9) > 1.0:
    warnings.warn(f"2024 CAPEX is ${total_2024:,.1f}bn; expected approximately $490.9bn.")

panel = capex_raw.rename(columns={"CAPEX": "capex_reported"}).copy()
panel["year"] = panel["quarter"].dt.year
panel["quarter_num"] = panel["quarter"].dt.quarter
panel["data_status"] = np.select(
    [
        panel["quarter"].le(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(),
        panel["quarter"].gt(LAST_ACTUAL_QUARTER) & panel["capex_reported"].notna(),
        panel["quarter"].le(LAST_ACTUAL_QUARTER),
    ],
    ["actual", "reported_estimate", "missing_actual"],
    default="missing_estimate",
)

baseline_lookup = panel.loc[
    panel["year"].eq(BASELINE_YEAR) & panel["capex_reported"].notna(),
    ["Company_Name", "quarter_num", "capex_reported"],
].rename(columns={"capex_reported": "baseline_same_quarter_2022_bn"})

panel = panel.merge(baseline_lookup, on=["Company_Name", "quarter_num"], how="left")
panel["is_baseline_hyperscaler"] = panel["Company_Name"].isin(BASELINE_HYPERSCALERS)
panel["raw_excess_2022_bn"] = panel["capex_reported"] - panel["baseline_same_quarter_2022_bn"]
panel["positive_excess_2022_bn"] = panel["raw_excess_2022_bn"].clip(lower=0)

panel["gross_ai_capex_central_bn"] = panel["capex_reported"]
panel.loc[panel["is_baseline_hyperscaler"], "gross_ai_capex_central_bn"] = panel.loc[
    panel["is_baseline_hyperscaler"], "positive_excess_2022_bn"
]
panel["baseline_source"] = np.where(
    panel["baseline_same_quarter_2022_bn"].notna(),
    "same_quarter_2022",
    "missing_same_quarter_2022",
)

panel["capital_profile"] = panel["Company_Name"].map(PROFILE_OVERRIDES)
panel["capital_profile"] = panel["capital_profile"].fillna(panel["GICS_Industry_Group"].map(INDUSTRY_PROFILE))
if panel["capital_profile"].isna().any():
    missing_profiles = sorted(panel.loc[panel["capital_profile"].isna(), "GICS_Industry_Group"].unique())
    raise ValueError(f"Missing capital-profile mappings for: {missing_profiles}")

model_panel = panel.loc[panel["quarter"].between(MODEL_START, ANALYSIS_END)].copy()

company_meta = panel.groupby("Company_Name", as_index=False).agg(GICS_Industry_Group=("GICS_Industry_Group", "first"))
baseline_stats = baseline_lookup.groupby("Company_Name").agg(
    valid_2022_quarters=("baseline_same_quarter_2022_bn", "count"),
    baseline_annual_2022_bn=("baseline_same_quarter_2022_bn", lambda values: values.sum(min_count=1)),
).reset_index()
baseline_audit = company_meta.merge(baseline_stats, on="Company_Name", how="left")
baseline_audit["valid_2022_quarters"] = baseline_audit["valid_2022_quarters"].fillna(0).astype(int)
baseline_audit["baseline_status"] = np.select(
    [baseline_audit["valid_2022_quarters"].eq(4), baseline_audit["valid_2022_quarters"].gt(0)],
    ["complete_2022", "partial_2022"],
    default="missing_2022",
)
baseline_audit["is_baseline_hyperscaler"] = baseline_audit["Company_Name"].isin(BASELINE_HYPERSCALERS)

coverage_audit = model_panel.groupby("quarter").agg(
    reported_companies=("capex_reported", lambda values: int(values.notna().sum())),
    reported_capex_bn=("capex_reported", lambda values: values.sum(min_count=1)),
    central_ai_companies=("gross_ai_capex_central_bn", lambda values: int(values.notna().sum())),
    central_ai_capex_bn=("gross_ai_capex_central_bn", lambda values: values.sum(min_count=1)),
    actual_companies=("data_status", lambda values: int(values.eq("actual").sum())),
    estimate_companies=("data_status", lambda values: int(values.eq("reported_estimate").sum())),
)
coverage_audit["central_coverage_share"] = coverage_audit["central_ai_capex_bn"].div(
    coverage_audit["reported_capex_bn"].replace(0, np.nan)
)

display(input_audit)
display(coverage_audit.loc[pd.Period("2025Q1", freq="Q"):ANALYSIS_END])


                                  metric       value
0                                   rows 22,339.0000
1                              companies    769.0000
2             duplicate company-quarters      0.0000
3                    blank industry rows      0.0000
4  companies with conflicting industries      0.0000
5                   missing CAPEX values  1,604.0000
6         negative reported CAPEX values    154.0000
7                       2024 CAPEX ($bn)    490.9156
         reported_companies  reported_capex_bn  central_ai_companies  central_ai_capex_bn  actual_companies  estimate_companies  central_coverage_share
quarter                                                                                                                                                
2025Q1                  423           141.0877                   423             104.3685               423                   0                  0.7397
2025Q2                  421           163.9058                   42

## 3. Allocation, GDP recognition, and nominal import subtraction

Imports are subtracted from recognized investment in nominal dollars. The central IT import formula excludes semiconductors. The literal power/cooling formula follows the Word framework even though its direction is the inverse of a conventional import-content ratio.

The compact `parse_period` and `bea_data` helpers follow the functions supplied in the framework document. The only additions are caching and explicit validation of series codes, labels, units, and dataset/table coverage.


In [3]:
def parse_period(value):
    return pd.Period(str(value), freq="Q")


_BEA_TABLE_CACHE = {}


def bea_data(table, dataset="NIPA"):
    """Return a BEA table with quarters on rows and SeriesCode values on columns."""
    key = (dataset, table)
    if key in _BEA_TABLE_CACHE:
        return _BEA_TABLE_CACHE[key]

    matching_bulk = {
        entry["bulk"] for entry in BEA_SERIES_REGISTRY.values() if entry["sheet"] == table
    }
    if len(matching_bulk) != 1:
        raise KeyError(f"Unable to identify one bulk source for {dataset}/{table}")
    bulk = matching_bulk.pop()
    expected_dataset = "NIUnderlyingDetail" if bulk == "underlying5" else "NIPA"
    if dataset != expected_dataset:
        raise ValueError(f"Dataset mismatch for {table}: expected {expected_dataset}, received {dataset}")

    filename, url = BEA_BULK_FILES[bulk]
    path = CACHE_DIR / filename
    if REFRESH_BEA or not path.exists():
        print(f"Downloading {filename} ...")
        urllib.request.urlretrieve(url, path)

    frame = pd.read_excel(path, sheet_name=table, header=7)
    frame = frame.rename(
        columns={frame.columns[0]: "Line", frame.columns[1]: "Description", frame.columns[2]: "SeriesCode"}
    )
    frame = frame.loc[frame["SeriesCode"].notna()].copy()
    frame["SeriesCode"] = frame["SeriesCode"].astype(str).str.strip()
    if frame["SeriesCode"].duplicated().any():
        duplicates = sorted(frame.loc[frame["SeriesCode"].duplicated(), "SeriesCode"].unique())
        unexpected = [code for code in duplicates if code != "ZZZZZZ"]
        if unexpected:
            raise ValueError(f"Duplicate series codes in {dataset}/{table}: {unexpected}")
        frame = frame.drop_duplicates("SeriesCode", keep="first")

    period_columns = [
        column for column in frame.columns if str(column)[:4].isdigit() and str(column)[4:5] == "Q"
    ]
    values = frame.set_index("SeriesCode")[period_columns].T.apply(pd.to_numeric, errors="coerce")
    values.index = pd.PeriodIndex([parse_period(column) for column in period_columns], freq="Q")
    values.index.name = "quarter"
    values.attrs["labels"] = frame.set_index("SeriesCode")["Description"].astype(str).str.strip().to_dict()
    values.attrs["source_url"] = url
    _BEA_TABLE_CACHE[key] = values.sort_index()
    return _BEA_TABLE_CACHE[key]


def official_series_from_registry(name):
    entry = BEA_SERIES_REGISTRY[name]
    dataset = "NIUnderlyingDetail" if entry["bulk"] == "underlying5" else "NIPA"
    table = bea_data(entry["sheet"], dataset)
    if entry["code"] not in table.columns:
        raise KeyError(f"Missing {entry['code']} in {dataset}/{entry['sheet']}")
    label = table.attrs["labels"].get(entry["code"], "")
    if entry["label"].lower() not in label.lower():
        raise ValueError(f"Series-label mismatch for {entry['code']}: {label!r}")
    series = (table[entry["code"]] * entry["scale"]).dropna().rename(name)
    audit = {
        "series": name,
        "code": entry["code"],
        "label": label,
        "table": entry["sheet"],
        "dataset": dataset,
        "units": entry["units"],
        "first_quarter": str(series.index.min()),
        "latest_quarter": str(series.index.max()),
        "source_url": table.attrs["source_url"],
    }
    return series, audit


official_series = {}
series_audit_rows = []
for series_name in BEA_SERIES_REGISTRY:
    series, audit = official_series_from_registry(series_name)
    official_series[series_name] = series
    series_audit_rows.append(audit)
series_audit = pd.DataFrame(series_audit_rows)


def allocate_and_recognize(source_panel, source_column, us_share, lag_weights):
    working = source_panel.loc[source_panel["quarter"].between(MODEL_START, ANALYSIS_END)].copy()
    working["scenario_ai_capex_bn"] = working[source_column]
    working["us_share"] = float(us_share)
    working["us_ai_capex_source_bn"] = working["scenario_ai_capex_bn"] * working["us_share"]

    allocation_blocks = []
    for category in CATEGORY_SPLITS.columns:
        block = working[
            [
                "Company_Name",
                "GICS_Industry_Group",
                "capital_profile",
                "quarter",
                "data_status",
                "scenario_ai_capex_bn",
                "us_ai_capex_source_bn",
            ]
        ].copy()
        block["category"] = category
        block["category_share"] = block["capital_profile"].map(CATEGORY_SPLITS[category])
        block["source_amount_bn"] = block["us_ai_capex_source_bn"] * block["category_share"]
        allocation_blocks.append(block)
    allocated = pd.concat(allocation_blocks, ignore_index=True)

    timed_blocks = []
    for category, weights in lag_weights.items():
        category_source = allocated.loc[allocated["category"].eq(category)].copy()
        for lag, weight in enumerate(weights):
            recognized = category_source.copy()
            recognized["source_quarter"] = recognized["quarter"]
            recognized["recorded_quarter"] = recognized["quarter"] + lag
            recognized["lag"] = lag
            recognized["lag_weight"] = weight
            recognized["gross_ai_investment_bn"] = recognized["source_amount_bn"] * weight
            timed_blocks.append(recognized)
    asset_detail = pd.concat(timed_blocks, ignore_index=True)
    tail_end = ANALYSIS_END + max(len(weights) - 1 for weights in lag_weights.values())
    asset_detail = asset_detail.loc[asset_detail["recorded_quarter"].le(tail_end)].copy()
    return working, allocated, asset_detail, tail_end


central_working, allocated_central, asset_detail_central, CENTRAL_TAIL_END = allocate_and_recognize(
    model_panel,
    "gross_ai_capex_central_bn",
    US_SHARE_DEFAULT,
    LAG_WEIGHTS,
)

import_series_names = [
    "import_capital_goods",
    "import_consumer_goods",
    "computer_imports",
    "telecom_imports",
    "electrical_imports",
    "computers_nominal",
    "communication_nominal",
    "electrical_nominal",
]
import_frame = pd.concat([official_series[name] for name in import_series_names], axis=1).dropna()
if len(import_frame) < IMPORT_SHARE_LOOKBACK_QUARTERS:
    raise ValueError("Insufficient complete BEA quarters for import-share calculations.")
import_window = import_frame.tail(IMPORT_SHARE_LOOKBACK_QUARTERS).copy()
import_window["computer_investment_weight"] = import_window["import_capital_goods"].div(
    import_window["import_capital_goods"] + import_window["import_consumer_goods"]
)
import_window["investment_assigned_computer_imports"] = (
    import_window["computer_imports"] * import_window["computer_investment_weight"]
)
import_window["it_import_ratio_raw"] = (
    import_window["investment_assigned_computer_imports"] + import_window["telecom_imports"]
).div(import_window["computers_nominal"] + import_window["communication_nominal"])
import_window["power_import_ratio_literal_raw"] = import_window["electrical_nominal"].div(
    import_window["electrical_imports"]
)
import_window["power_import_ratio_conventional_raw"] = import_window["electrical_imports"].div(
    import_window["electrical_nominal"]
)

central_it_import_share_raw = float(import_window["it_import_ratio_raw"].mean())
central_power_import_share_raw = float(import_window["power_import_ratio_literal_raw"].mean())
conventional_power_import_share_raw = float(import_window["power_import_ratio_conventional_raw"].mean())

CENTRAL_IMPORT_SHARES = {
    "dc_structures": 0.00,
    "power_structures": 0.00,
    "it_equipment": float(np.clip(central_it_import_share_raw, 0.0, 1.0)),
    "power_cooling_equipment": float(np.clip(central_power_import_share_raw, 0.0, 1.0)),
}
CONVENTIONAL_POWER_IMPORT_SHARES = {
    **CENTRAL_IMPORT_SHARES,
    "power_cooling_equipment": float(np.clip(conventional_power_import_share_raw, 0.0, 1.0)),
}

import_diagnostics = import_window.reset_index(names="quarter")
import_diagnostics["quarter"] = import_diagnostics["quarter"].astype(str)
import_diagnostics["it_import_ratio_clipped"] = import_diagnostics["it_import_ratio_raw"].clip(0, 1)
import_diagnostics["power_import_ratio_literal_clipped"] = import_diagnostics[
    "power_import_ratio_literal_raw"
].clip(0, 1)
average_row = {
    "quarter": "Two-quarter average",
    **{column: import_window[column].mean() for column in import_window.columns},
    "it_import_ratio_clipped": CENTRAL_IMPORT_SHARES["it_equipment"],
    "power_import_ratio_literal_clipped": CENTRAL_IMPORT_SHARES["power_cooling_equipment"],
}
import_diagnostics = pd.concat([import_diagnostics, pd.DataFrame([average_row])], ignore_index=True)


def build_nominal_results(asset_detail, import_shares):
    detail = asset_detail.copy()
    detail["import_share"] = detail["category"].map(import_shares)
    if detail["import_share"].isna().any():
        missing = sorted(detail.loc[detail["import_share"].isna(), "category"].unique())
        raise ValueError(f"Missing import shares for {missing}")
    detail["associated_imports_bn"] = detail["gross_ai_investment_bn"] * detail["import_share"]
    detail["net_ai_gdp_flow_bn"] = detail["gross_ai_investment_bn"] - detail["associated_imports_bn"]

    asset_quarter = detail.groupby(["recorded_quarter", "category"], as_index=False).agg(
        gross_ai_investment_bn=("gross_ai_investment_bn", lambda values: values.sum(min_count=1)),
        associated_imports_bn=("associated_imports_bn", lambda values: values.sum(min_count=1)),
        net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", lambda values: values.sum(min_count=1)),
    )
    quarterly = asset_quarter.groupby("recorded_quarter").agg(
        gross_ai_investment_bn=("gross_ai_investment_bn", lambda values: values.sum(min_count=1)),
        associated_imports_bn=("associated_imports_bn", lambda values: values.sum(min_count=1)),
        net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", lambda values: values.sum(min_count=1)),
    )
    quarterly.index.name = "quarter"
    annual = quarterly.groupby(quarterly.index.year).agg(
        gross_ai_investment_bn=("gross_ai_investment_bn", lambda values: values.sum(min_count=1)),
        associated_imports_bn=("associated_imports_bn", lambda values: values.sum(min_count=1)),
        net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", lambda values: values.sum(min_count=1)),
    )
    annual.index.name = "year"
    return detail, asset_quarter, quarterly, annual


central_detail, central_asset_quarter, central_quarterly_flows, central_annual_flows = build_nominal_results(
    asset_detail_central,
    CENTRAL_IMPORT_SHARES,
)

source_quarterly = central_working.groupby("quarter").agg(
    company_capex_bn=("capex_reported", lambda values: values.sum(min_count=1)),
    gross_ai_capex_bn=("scenario_ai_capex_bn", lambda values: values.sum(min_count=1)),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", lambda values: values.sum(min_count=1)),
)
quarterly_results = source_quarterly.join(central_quarterly_flows, how="outer")
quarterly_results["data_status"] = np.where(quarterly_results.index <= LAST_ACTUAL_QUARTER, "Actual", "Estimate")

annual_results = quarterly_results.groupby(quarterly_results.index.year).agg(
    company_capex_bn=("company_capex_bn", lambda values: values.sum(min_count=1)),
    gross_ai_capex_bn=("gross_ai_capex_bn", lambda values: values.sum(min_count=1)),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", lambda values: values.sum(min_count=1)),
    gross_ai_investment_bn=("gross_ai_investment_bn", lambda values: values.sum(min_count=1)),
    associated_imports_bn=("associated_imports_bn", lambda values: values.sum(min_count=1)),
    net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", lambda values: values.sum(min_count=1)),
)
annual_results.index.name = "year"
annual_results["data_status"] = np.select(
    [annual_results.index <= 2025, annual_results.index == 2026],
    ["Actual", "Actual through Q2 / estimate thereafter"],
    default="Estimate",
)

industry_breakdown = central_working.groupby(
    [central_working["quarter"].dt.year, "GICS_Industry_Group"]
).agg(
    gross_ai_capex_bn=("scenario_ai_capex_bn", lambda values: values.sum(min_count=1)),
    us_ai_capex_source_bn=("us_ai_capex_source_bn", lambda values: values.sum(min_count=1)),
).reset_index().rename(columns={"quarter": "year"})

asset_breakdown = central_asset_quarter.copy()
asset_breakdown["year"] = asset_breakdown["recorded_quarter"].dt.year
asset_breakdown = asset_breakdown.groupby(["year", "category"], as_index=False).agg(
    gross_ai_investment_bn=("gross_ai_investment_bn", lambda values: values.sum(min_count=1)),
    associated_imports_bn=("associated_imports_bn", lambda values: values.sum(min_count=1)),
    net_ai_gdp_flow_bn=("net_ai_gdp_flow_bn", lambda values: values.sum(min_count=1)),
)

print("Central import shares")
display(pd.Series(CENTRAL_IMPORT_SHARES, name="import_share").map(lambda value: f"{value:.2%}"))
display(import_diagnostics[["quarter", "computer_investment_weight", "it_import_ratio_raw", "power_import_ratio_literal_raw", "power_import_ratio_conventional_raw"]])
display(annual_results.loc[2023:2028])


Central import shares
dc_structures               0.00%
power_structures            0.00%
it_equipment               93.74%
power_cooling_equipment    75.32%
               quarter  computer_investment_weight  it_import_ratio_raw  power_import_ratio_literal_raw  power_import_ratio_conventional_raw
0               2026Q1                      0.6819               0.9262                          0.7770                               1.2869
1               2026Q2                      0.6860               0.9486                          0.7294                               1.3710
2  Two-quarter average                      0.6840               0.9374                          0.7532                               1.3290
      company_capex_bn  gross_ai_capex_bn  us_ai_capex_source_bn  gross_ai_investment_bn  associated_imports_bn  net_ai_gdp_flow_bn                              data_status
year                                                                                                     

## 4. Gross and net chained deflators

Gross and net nominal investment are expanded into five price components. Each aggregate receives its own Törnqvist-style chained price index using the framework's `chained_weights`, `chained_dlog_price`, `matched_dlog_nominal`, and `chain_from_dlog` functions. The net real series is deflated directly from nominal net investment.


In [4]:
actual_index = pd.period_range(MODEL_START, REAL_CONTRIBUTION_END, freq="Q")

computers_nominal = official_series["computers_nominal"].reindex(actual_index)
communication_nominal = official_series["communication_nominal"].reindex(actual_index)
it_nominal_total = computers_nominal + communication_nominal
computer_mix = computers_nominal.div(it_nominal_total.replace(0, np.nan))
communication_mix = 1.0 - computer_mix

component_prices = pd.DataFrame(
    {
        "data_centers": official_series["dc_structures_price"].reindex(actual_index),
        "power_structures": official_series["power_structures_price"].reindex(actual_index),
        "computers": official_series["computers_price"].reindex(actual_index),
        "communication_equipment": official_series["communication_price"].reindex(actual_index),
        "electrical_apparatus": official_series["electrical_price"].reindex(actual_index),
    },
    index=actual_index,
)

if component_prices.isna().any().any() or computer_mix.isna().any():
    raise ValueError("Missing actual-quarter investment price or IT nominal-mix data.")


def expand_asset_components(asset_quarter, value_column):
    pivot = asset_quarter.pivot(index="recorded_quarter", columns="category", values=value_column)
    pivot = pivot.reindex(actual_index)
    required = list(CATEGORY_SPLITS.columns)
    missing_columns = set(required).difference(pivot.columns)
    if missing_columns:
        raise ValueError(f"Missing asset categories in {value_column}: {sorted(missing_columns)}")
    expanded = pd.DataFrame(index=actual_index)
    expanded["data_centers"] = pivot["dc_structures"]
    expanded["power_structures"] = pivot["power_structures"]
    expanded["computers"] = pivot["it_equipment"] * computer_mix
    expanded["communication_equipment"] = pivot["it_equipment"] * communication_mix
    expanded["electrical_apparatus"] = pivot["power_cooling_equipment"]
    if expanded.isna().any().any():
        missing_rows = expanded.index[expanded.isna().any(axis=1)].astype(str).tolist()
        raise ValueError(f"Missing nominal expanded components for {value_column}: {missing_rows}")
    aggregate = expanded.sum(axis=1)
    if (aggregate < 0).any():
        bad = expanded.index[aggregate < 0].astype(str).tolist()
        raise ValueError(f"Negative aggregate nominal {value_column}: {bad}")
    if not aggregate.gt(0).any():
        raise ValueError(f"No positive aggregate nominal values for {value_column}")
    return expanded


def chained_weights(nominal, prices):
    """Two-period average nominal shares, as specified in the framework document."""
    valid = nominal.notna() & prices.notna()
    matched = valid & valid.shift(1, fill_value=False)
    now = nominal.where(matched)
    previous = nominal.shift(1).where(matched)
    share_now = now.div(now.sum(axis=1, min_count=1).replace(0, np.nan), axis=0)
    share_previous = previous.div(previous.sum(axis=1, min_count=1).replace(0, np.nan), axis=0)
    return 0.5 * (share_now + share_previous), matched


def chained_dlog_price(nominal, prices):
    weights, matched = chained_weights(nominal, prices)
    dlog_prices = np.log(prices).diff().where(matched)
    return (weights * dlog_prices).sum(axis=1, min_count=1), weights, matched


def matched_dlog_nominal(nominal, matched):
    now = nominal.where(matched).sum(axis=1, min_count=1)
    previous = nominal.shift(1).where(matched).sum(axis=1, min_count=1)
    return np.log(now) - np.log(previous)


def chain_from_dlog(dlog, initial_level):
    """Chain forward without treating a missing intermediate price change as zero."""
    if dlog.iloc[1:].isna().any():
        missing = dlog.index[dlog.isna()][1:].astype(str).tolist()
        raise ValueError(f"Missing chained price changes: {missing}")
    growth = dlog.copy()
    growth.iloc[0] = 0.0
    return initial_level * np.exp(growth.cumsum())


def build_chained_deflator(nominal_components, prices, basis):
    if not nominal_components.index.equals(prices.index):
        raise ValueError(f"Index mismatch in {basis} deflator")
    if list(nominal_components.columns) != list(prices.columns):
        raise ValueError(f"Component mismatch in {basis} deflator")
    if prices.isna().any().any() or (prices <= 0).any().any():
        raise ValueError(f"Invalid prices in {basis} deflator")

    nominal_total = nominal_components.sum(axis=1, min_count=1)
    if nominal_total.isna().any() or (nominal_total < 0).any():
        raise ValueError(f"Invalid nominal totals in {basis} deflator")
    positive_quarters = nominal_total.gt(0)
    if not positive_quarters.any():
        raise ValueError(f"No positive nominal totals in {basis} deflator")
    first_quarter = nominal_total.index[positive_quarters][0]
    active_index = nominal_total.loc[first_quarter:].index
    if (nominal_total.loc[active_index] <= 0).any():
        bad = nominal_total.loc[active_index].index[nominal_total.loc[active_index] <= 0].astype(str).tolist()
        raise ValueError(f"Nonpositive nominal total after {basis} chain begins: {bad}")

    shares = nominal_components.div(nominal_total.replace(0, np.nan), axis=0)
    active_nominal = nominal_components.loc[active_index]
    active_prices = prices.loc[active_index]
    active_dlog, active_weights, matched = chained_dlog_price(active_nominal, active_prices)
    nominal_dlog = matched_dlog_nominal(active_nominal, matched)
    if nominal_dlog.iloc[1:].isna().any():
        raise ValueError(f"Missing matched nominal changes in {basis} deflator")

    anchor = float(np.exp((shares.loc[first_quarter] * np.log(prices.loc[first_quarter])).sum()))
    active_deflator = chain_from_dlog(active_dlog, anchor)
    deflator = active_deflator.reindex(nominal_total.index).rename(f"{basis}_deflator")
    dlog_deflator = active_dlog.reindex(nominal_total.index)
    two_period_weights = active_weights.reindex(nominal_total.index)

    reproduced_dlog = np.log(deflator).diff()
    dlog_error = float((reproduced_dlog - dlog_deflator).dropna().abs().max())
    real_quarterly = pd.Series(0.0, index=nominal_total.index)
    real_quarterly.loc[active_index] = nominal_total.loc[active_index].div(deflator.loc[active_index] / 100.0)
    output = pd.DataFrame(
        {
            "nominal_total_bn": nominal_total,
            "deflator": deflator,
            "real_quarterly_bn": real_quarterly,
            "real_saar_bn": real_quarterly * 4.0,
            "dlog_deflator": dlog_deflator,
        }
    )
    return output, shares, two_period_weights, dlog_error


real_gdp = official_series["real_gdp"].reindex(actual_index)
if real_gdp.isna().any():
    raise ValueError("Real GDP is missing for one or more requested actual quarters.")


def qoq_annualized_growth(level):
    return ((level / level.shift(1)) ** QOQ_ANNUALIZATION_EXPONENT - 1.0) * 100.0


def yoy_growth(level):
    return (level / level.shift(4) - 1.0) * 100.0


def build_real_contributions(asset_quarter):
    gross_components = expand_asset_components(asset_quarter, "gross_ai_investment_bn")
    net_components = expand_asset_components(asset_quarter, "net_ai_gdp_flow_bn")
    gross_deflator, gross_shares, gross_weights, gross_error = build_chained_deflator(
        gross_components, component_prices, "gross"
    )
    net_deflator, net_shares, net_weights, net_error = build_chained_deflator(
        net_components, component_prices, "net"
    )

    result = pd.DataFrame(index=actual_index)
    result.index.name = "quarter"
    result["real_gdp_bn_saar"] = real_gdp
    result["gross_ai_real_saar_bn"] = gross_deflator["real_saar_bn"]
    result["net_ai_real_saar_bn"] = net_deflator["real_saar_bn"]
    result["real_gdp_growth_qoq_saar_pct"] = qoq_annualized_growth(real_gdp)
    result["real_gdp_growth_yoy_pct"] = yoy_growth(real_gdp)

    gross_counterfactual = real_gdp - result["gross_ai_real_saar_bn"]
    net_counterfactual = real_gdp - result["net_ai_real_saar_bn"]
    if (gross_counterfactual <= 0).any() or (net_counterfactual <= 0).any():
        raise ValueError("AI counterfactual produced nonpositive real GDP.")

    result["real_gdp_growth_without_gross_ai_qoq_saar_pct"] = qoq_annualized_growth(gross_counterfactual)
    result["real_gdp_growth_without_net_ai_qoq_saar_pct"] = qoq_annualized_growth(net_counterfactual)
    result["real_gross_ai_contribution_pp"] = (
        result["real_gdp_growth_qoq_saar_pct"]
        - result["real_gdp_growth_without_gross_ai_qoq_saar_pct"]
    )
    result["real_net_ai_contribution_pp"] = (
        result["real_gdp_growth_qoq_saar_pct"]
        - result["real_gdp_growth_without_net_ai_qoq_saar_pct"]
    )
    result["implied_import_effect_pp"] = (
        result["real_net_ai_contribution_pp"] - result["real_gross_ai_contribution_pp"]
    )

    result["real_gdp_growth_without_gross_ai_yoy_pct"] = yoy_growth(gross_counterfactual)
    result["real_gdp_growth_without_net_ai_yoy_pct"] = yoy_growth(net_counterfactual)
    result["real_gross_ai_contribution_yoy_pp"] = (
        result["real_gdp_growth_yoy_pct"] - result["real_gdp_growth_without_gross_ai_yoy_pct"]
    )
    result["real_net_ai_contribution_yoy_pp"] = (
        result["real_gdp_growth_yoy_pct"] - result["real_gdp_growth_without_net_ai_yoy_pct"]
    )
    result["implied_import_effect_yoy_pp"] = (
        result["real_net_ai_contribution_yoy_pp"] - result["real_gross_ai_contribution_yoy_pp"]
    )
    result["net_ai_share_of_real_gdp_growth_pct"] = (
        result["real_net_ai_contribution_pp"] / result["real_gdp_growth_qoq_saar_pct"] * 100.0
    )
    result["growth_share_warning"] = np.where(
        result["real_gdp_growth_qoq_saar_pct"].abs().lt(0.5),
        "Unstable denominator: |GDP growth| < 0.5%",
        "",
    )
    return {
        "contributions": result,
        "gross_deflator": gross_deflator,
        "net_deflator": net_deflator,
        "gross_shares": gross_shares,
        "net_shares": net_shares,
        "gross_weights": gross_weights,
        "net_weights": net_weights,
        "gross_dlog_error": gross_error,
        "net_dlog_error": net_error,
    }


central_real = build_real_contributions(central_asset_quarter)
real_contributions = central_real["contributions"]

display(real_contributions.loc[pd.Period("2024Q1", freq="Q"):REAL_CONTRIBUTION_END, [
    "real_gdp_growth_qoq_saar_pct",
    "real_gross_ai_contribution_pp",
    "implied_import_effect_pp",
    "real_net_ai_contribution_pp",
    "real_net_ai_contribution_yoy_pp",
]])


         real_gdp_growth_qoq_saar_pct  real_gross_ai_contribution_pp  implied_import_effect_pp  real_net_ai_contribution_pp  real_net_ai_contribution_yoy_pp
quarter                                                                                                                                                     
2024Q1                         0.8421                         0.1306                   -0.0723                       0.0582                           0.0567
2024Q2                         3.5893                         0.1832                   -0.1302                       0.0529                           0.0549
2024Q3                         3.3402                         0.1994                   -0.1536                       0.0458                           0.0544
2024Q4                         1.8522                         0.5711                   -0.3893                       0.1818                           0.0849
2025Q1                        -0.6485                     

## 5. One-at-a-time sensitivity scenarios

The central case remains the literal framework selected by the user. Each scenario changes one assumption and reruns attribution, recognition, nominal import netting, composite deflation, and the GDP counterfactual.


In [5]:
# Build profile-specific price levels for the inflation-indexed 2022 baseline scenario.
category_price_actual = pd.DataFrame(index=actual_index)
category_price_actual["dc_structures"] = component_prices["data_centers"]
category_price_actual["power_structures"] = component_prices["power_structures"]
category_price_actual["it_equipment"] = np.exp(
    computer_mix * np.log(component_prices["computers"])
    + communication_mix * np.log(component_prices["communication_equipment"])
)
category_price_actual["power_cooling_equipment"] = component_prices["electrical_apparatus"]


def extend_price_series(series, end_quarter, drift_quarters):
    extended = series.copy().sort_index()
    log_changes = np.log(extended).diff().dropna().tail(drift_quarters)
    drift = float(log_changes.mean())
    for quarter in pd.period_range(extended.index.max() + 1, end_quarter, freq="Q"):
        extended.loc[quarter] = extended.iloc[-1] * math.exp(drift)
    return extended


category_price_full = pd.DataFrame(index=pd.period_range(MODEL_START, ANALYSIS_END, freq="Q"))
for category in category_price_actual.columns:
    category_price_full[category] = extend_price_series(
        category_price_actual[category].dropna(), ANALYSIS_END, PROFILE_PRICE_DRIFT_QUARTERS
    ).reindex(category_price_full.index)

profile_price_levels = {}
for profile, splits in CATEGORY_SPLITS.iterrows():
    log_level = sum(
        float(splits[category]) * np.log(category_price_full[category])
        for category in CATEGORY_SPLITS.columns
        if splits[category] > 0
    )
    profile_price_levels[profile] = np.exp(log_level)

inflation_indexed_ai = panel["capex_reported"].copy()
for index, row in panel.loc[panel["is_baseline_hyperscaler"]].iterrows():
    if pd.isna(row["capex_reported"]) or pd.isna(row["baseline_same_quarter_2022_bn"]):
        inflation_indexed_ai.loc[index] = np.nan
        continue
    current_quarter = row["quarter"]
    baseline_quarter = pd.Period(year=BASELINE_YEAR, quarter=row["quarter_num"], freq="Q")
    if current_quarter < MODEL_START:
        inflation_indexed_ai.loc[index] = row["positive_excess_2022_bn"]
        continue
    profile = row["capital_profile"]
    multiplier = profile_price_levels[profile].loc[current_quarter] / profile_price_levels[profile].loc[baseline_quarter]
    indexed_baseline = row["baseline_same_quarter_2022_bn"] * multiplier
    inflation_indexed_ai.loc[index] = max(row["capex_reported"] - indexed_baseline, 0.0)
panel["gross_ai_capex_inflation_baseline_bn"] = inflation_indexed_ai

panel["gross_ai_capex_baseline_all_bn"] = panel["positive_excess_2022_bn"]
panel["gross_ai_capex_hyperscaler_only_bn"] = np.where(
    panel["is_baseline_hyperscaler"], panel["positive_excess_2022_bn"], 0.0
)

IMMEDIATE_LAGS = {category: [1.0] for category in LAG_WEIGHTS}
DELAYED_LAGS = {category: [0.0] + list(weights) for category, weights in LAG_WEIGHTS.items()}


def run_scenario(name, source_column, us_share, import_shares, lag_weights):
    _, _, detail, tail_end = allocate_and_recognize(panel, source_column, us_share, lag_weights)
    _, asset_quarter, quarterly, annual = build_nominal_results(detail, import_shares)
    real = build_real_contributions(asset_quarter)
    latest = real["contributions"].loc[REAL_CONTRIBUTION_END]
    recognition_method = (
        "Immediate (100% in source quarter)"
        if lag_weights == IMMEDIATE_LAGS
        else "One-quarter delay applied to central lags"
        if lag_weights == DELAYED_LAGS
        else "Central asset-specific lag weights"
    )
    return {
        "scenario": name,
        "source_method": source_column,
        "us_share": us_share,
        "it_import_share": import_shares["it_equipment"],
        "power_cooling_import_share": import_shares["power_cooling_equipment"],
        "recognition_method": recognition_method,
        "tail_end": str(tail_end),
        "2025_gross_nominal_bn": float(annual.loc[2025, "gross_ai_investment_bn"]),
        "2025_net_nominal_bn": float(annual.loc[2025, "net_ai_gdp_flow_bn"]),
        "2026_gross_nominal_bn": float(annual.loc[2026, "gross_ai_investment_bn"]),
        "2026_net_nominal_bn": float(annual.loc[2026, "net_ai_gdp_flow_bn"]),
        "latest_gross_contribution_pp": float(latest["real_gross_ai_contribution_pp"]),
        "latest_net_contribution_pp": float(latest["real_net_ai_contribution_pp"]),
        "latest_implied_import_effect_pp": float(latest["implied_import_effect_pp"]),
        "cumulative_recognized_gross_bn": float(quarterly["gross_ai_investment_bn"].sum(min_count=1)),
        "cumulative_nominal_net_bn": float(quarterly["net_ai_gdp_flow_bn"].sum(min_count=1)),
    }


scenario_specs = [
    ("Central literal framework", "gross_ai_capex_central_bn", 0.70, CENTRAL_IMPORT_SHARES, LAG_WEIGHTS),
    ("Positive 2022 excess for all", "gross_ai_capex_baseline_all_bn", 0.70, CENTRAL_IMPORT_SHARES, LAG_WEIGHTS),
    ("Five hyperscalers only", "gross_ai_capex_hyperscaler_only_bn", 0.70, CENTRAL_IMPORT_SHARES, LAG_WEIGHTS),
    ("Inflation-indexed hyperscaler baseline", "gross_ai_capex_inflation_baseline_bn", 0.70, CENTRAL_IMPORT_SHARES, LAG_WEIGHTS),
    ("US share 60%", "gross_ai_capex_central_bn", 0.60, CENTRAL_IMPORT_SHARES, LAG_WEIGHTS),
    ("US share 80%", "gross_ai_capex_central_bn", 0.80, CENTRAL_IMPORT_SHARES, LAG_WEIGHTS),
    ("V2 fixed import shares", "gross_ai_capex_central_bn", 0.70, V2_FIXED_IMPORT_SHARES, LAG_WEIGHTS),
    ("Conventional power import ratio", "gross_ai_capex_central_bn", 0.70, CONVENTIONAL_POWER_IMPORT_SHARES, LAG_WEIGHTS),
    ("Immediate recognition", "gross_ai_capex_central_bn", 0.70, CENTRAL_IMPORT_SHARES, IMMEDIATE_LAGS),
    ("One-quarter delayed recognition", "gross_ai_capex_central_bn", 0.70, CENTRAL_IMPORT_SHARES, DELAYED_LAGS),
]

scenario_comparison = pd.DataFrame(
    [run_scenario(name, source, us_share, imports, lags) for name, source, us_share, imports, lags in scenario_specs]
)
display(scenario_comparison)


                                 scenario                         source_method  us_share  it_import_share  power_cooling_import_share                         recognition_method tail_end  2025_gross_nominal_bn  2025_net_nominal_bn  2026_gross_nominal_bn  2026_net_nominal_bn  latest_gross_contribution_pp  latest_net_contribution_pp  latest_implied_import_effect_pp  cumulative_recognized_gross_bn  cumulative_nominal_net_bn
0               Central literal framework             gross_ai_capex_central_bn    0.7000           0.9374                      0.7532         Central asset-specific lag weights   2029Q3               343.3816             168.2482               611.7054             261.2982                        0.6559                      0.2851                          -0.3708                      3,379.3070                 1,574.7093
1            Positive 2022 excess for all        gross_ai_capex_baseline_all_bn    0.7000           0.9374                      0.7532         Central

## 6. Reconciliation checks and workbook export

The exported workbook preserves the familiar v2 detail sheets and adds explicit import, deflator, weight, scenario, and summary tabs. The workbook is authored with the bundled spreadsheet runtime and contains linked summary formulas.


In [6]:
# Central output tables
company_output = central_working[
    [
        "quarter",
        "Company_Name",
        "GICS_Industry_Group",
        "capital_profile",
        "data_status",
        "capex_reported",
        "baseline_same_quarter_2022_bn",
        "baseline_source",
        "is_baseline_hyperscaler",
        "scenario_ai_capex_bn",
        "us_share",
        "us_ai_capex_source_bn",
    ]
].copy()
company_output = company_output.rename(columns={"scenario_ai_capex_bn": "gross_ai_capex_bn"})

quarterly_output = quarterly_results.reset_index()
annual_output = annual_results.reset_index()
real_output = real_contributions.reset_index()
coverage_output = coverage_audit.reset_index()

gross_deflator_output = central_real["gross_deflator"].reset_index().rename(
    columns={
        "index": "quarter",
        "nominal_total_bn": "gross_nominal_total_bn",
        "real_quarterly_bn": "gross_real_quarterly_bn",
        "real_saar_bn": "gross_real_saar_bn",
    }
)
net_deflator_output = central_real["net_deflator"].reset_index().rename(
    columns={
        "index": "quarter",
        "nominal_total_bn": "net_nominal_total_bn",
        "real_quarterly_bn": "net_real_quarterly_bn",
        "real_saar_bn": "net_real_saar_bn",
    }
)

weight_blocks = []
for basis in ["gross", "net"]:
    shares = central_real[f"{basis}_shares"]
    weights = central_real[f"{basis}_weights"]
    for component in shares.columns:
        block = pd.DataFrame(
            {
                "quarter": shares.index,
                "basis": basis,
                "component": component,
                "nominal_share": shares[component],
                "two_period_weight": weights[component],
                "component_price_index": component_prices[component],
            }
        )
        weight_blocks.append(block)
deflator_weights_output = pd.concat(weight_blocks, ignore_index=True)

assumption_rows = [
    ("LAST_ACTUAL_QUARTER", str(LAST_ACTUAL_QUARTER), "Model date"),
    ("MODEL_START", str(MODEL_START), "Model date"),
    ("ANALYSIS_END", str(ANALYSIS_END), "Model date"),
    ("BASELINE_HYPERSCALERS", json.dumps(sorted(BASELINE_HYPERSCALERS)), "AI attribution"),
    ("NON_HYPERSCALER_ATTRIBUTION", "100% of reported CAPEX", "AI attribution"),
    ("HYPERSCALER_BASELINE", "Positive excess over same calendar quarter of 2022", "AI attribution"),
    ("MISSING_DATA", "Leave missing; no interpolation or zero fill", "Data treatment"),
    ("NEGATIVE_REPORTED_CAPEX", "Retain signed values as reported in the literal central case", "Data treatment"),
    ("US_SHARE_DEFAULT", US_SHARE_DEFAULT, "Geography"),
    ("IMPORT_SHARE_LOOKBACK_QUARTERS", IMPORT_SHARE_LOOKBACK_QUARTERS, "Imports"),
    ("IT_IMPORT_FORMULA", "(B852RC * A650RC/(A650RC+A652RC) + LA001146) / (B935RC+C275RC)", "Imports"),
    ("SEMICONDUCTOR_IMPORTS", "Excluded", "Imports"),
    ("POWER_IMPORT_FORMULA_LITERAL", "C276RC / LA001149", "Imports"),
    ("CENTRAL_IT_IMPORT_SHARE", CENTRAL_IMPORT_SHARES["it_equipment"], "Imports"),
    ("CENTRAL_POWER_COOLING_IMPORT_SHARE", CENTRAL_IMPORT_SHARES["power_cooling_equipment"], "Imports"),
    ("REAL_NET_METHOD", "Deflate nominal net directly with its own chained deflator", "Real calculation"),
    ("REAL_ADDIVITY", "Do not calculate real net as real gross minus real imports", "Real calculation"),
    ("PROFILE_PRICE_DRIFT_QUARTERS", PROFILE_PRICE_DRIFT_QUARTERS, "Inflation-baseline scenario"),
]
for profile, row in CATEGORY_SPLITS.iterrows():
    for category, value in row.items():
        assumption_rows.append((f"CATEGORY_SPLIT.{profile}.{category}", value, "Asset allocation"))
for category, weights in LAG_WEIGHTS.items():
    assumption_rows.append((f"LAG_WEIGHTS.{category}", json.dumps(weights), "GDP recognition"))
for category, value in CENTRAL_IMPORT_SHARES.items():
    assumption_rows.append((f"IMPORT_SHARE.{category}", value, "Imports"))
for company, profile in PROFILE_OVERRIDES.items():
    assumption_rows.append((f"PROFILE_OVERRIDE.{company}", profile, "Capital profile"))
assumptions_output = pd.DataFrame(assumption_rows, columns=["assumption", "value", "category"])

# Reconciliation calculations
hyper_2022_rows = panel[
    panel["is_baseline_hyperscaler"]
    & panel["year"].eq(2022)
    & panel["baseline_same_quarter_2022_bn"].notna()
]
same_quarter_2022_error = float(hyper_2022_rows["raw_excess_2022_bn"].abs().max())
nonhyper_mask = (~panel["is_baseline_hyperscaler"]) & panel["capex_reported"].notna()
nonhyper_attribution_error = float(
    (panel.loc[nonhyper_mask, "gross_ai_capex_central_bn"] - panel.loc[nonhyper_mask, "capex_reported"]).abs().max()
)

us_source_total = float(central_working["us_ai_capex_source_bn"].sum(min_count=1))
allocated_total = float(allocated_central["source_amount_bn"].sum(min_count=1))
recognized_total = float(asset_detail_central["gross_ai_investment_bn"].sum(min_count=1))
nominal_identity_error = float(
    (
        central_detail["net_ai_gdp_flow_bn"]
        - central_detail["gross_ai_investment_bn"]
        + central_detail["associated_imports_bn"]
    ).abs().max()
)
nonnegative_investment_rows = central_detail["gross_ai_investment_bn"].ge(0)
imports_over_investment = max(
    0.0,
    float(
        (
            central_detail.loc[nonnegative_investment_rows, "associated_imports_bn"]
            - central_detail.loc[nonnegative_investment_rows, "gross_ai_investment_bn"]
        ).max()
    ),
)
import_share_bound_error = max(
    0.0,
    max(CENTRAL_IMPORT_SHARES.values()) - 1.0,
    -min(CENTRAL_IMPORT_SHARES.values()),
)

gross_weight_error = float((central_real["gross_shares"].sum(axis=1) - 1.0).abs().max())
net_weight_error = float((central_real["net_shares"].sum(axis=1) - 1.0).abs().max())
price_missing_count = int(component_prices.isna().sum().sum())
counterfactual_minimum = float(
    pd.concat(
        [
            real_gdp - real_contributions["gross_ai_real_saar_bn"],
            real_gdp - real_contributions["net_ai_real_saar_bn"],
        ],
        axis=1,
    ).min().min()
)
registry_text = json.dumps(BEA_SERIES_REGISTRY, sort_keys=True)
semiconductor_code_count = sum(code in registry_text for code in FORBIDDEN_SEMICONDUCTOR_CODES)

expected_ratio_checks = {
    "2026Q1": 0.926151,
    "2026Q2": 0.948593,
    "Two-quarter average": 0.937372,
}
actual_ratio_checks = {
    str(import_window.index[0]): float(import_window["it_import_ratio_raw"].iloc[0]),
    str(import_window.index[1]): float(import_window["it_import_ratio_raw"].iloc[1]),
    "Two-quarter average": central_it_import_share_raw,
}
ratio_vintage_difference = max(
    abs(actual_ratio_checks[key] - expected_ratio_checks[key])
    for key in expected_ratio_checks
)

central_cumulative = float(scenario_comparison.loc[
    scenario_comparison["scenario"].eq("Central literal framework"), "cumulative_recognized_gross_bn"
].iloc[0])
immediate_cumulative = float(scenario_comparison.loc[
    scenario_comparison["scenario"].eq("Immediate recognition"), "cumulative_recognized_gross_bn"
].iloc[0])
delayed_cumulative = float(scenario_comparison.loc[
    scenario_comparison["scenario"].eq("One-quarter delayed recognition"), "cumulative_recognized_gross_bn"
].iloc[0])
scenario_conservation_error = max(
    abs(central_cumulative - immediate_cumulative),
    abs(central_cumulative - delayed_cumulative),
)

check_specs = [
    ("Category splits sum to 100%", float((CATEGORY_SPLITS.sum(axis=1) - 1).abs().max()), 0.0, 1e-12, "Every profile fully allocates CAPEX."),
    ("Lag weights sum to 100%", max(abs(sum(weights) - 1) for weights in LAG_WEIGHTS.values()), 0.0, 1e-12, "Recognition conserves each asset dollar."),
    ("Matched hyperscaler 2022 quarters have zero excess", same_quarter_2022_error, 0.0, 1e-12, "Same-quarter baseline identity."),
    ("Non-hyperscaler attribution equals reported CAPEX", nonhyper_attribution_error, 0.0, 1e-12, "Central literal-framework perimeter."),
    ("US source allocation reconciles", allocated_total - us_source_total, 0.0, 1e-8, "Four categories sum to the US source amount."),
    ("Recognition timing conserves dollars through tail", recognized_total - allocated_total, 0.0, 1e-8, "Tail extends through the longest lag."),
    ("Central import shares remain within [0,1]", import_share_bound_error, 0.0, 1e-12, "Literal ratios are clipped before application."),
    ("Imports do not exceed nonnegative nominal investment", imports_over_investment, 0.0, 1e-12, "Negative reported CAPEX values retain their proportional signed treatment."),
    ("Nominal net equals gross minus imports", nominal_identity_error, 0.0, 1e-12, "Nominal identity only; no real additivity assertion."),
    ("Semiconductor codes absent", float(semiconductor_code_count), 0.0, 0.0, "LA001145 and IA001145 are intentionally excluded."),
    ("Current-vintage IT import ratios match specification", ratio_vintage_difference, 0.0, 5e-6, "2026Q1, 2026Q2, and two-quarter mean."),
    ("Gross nominal shares sum to 100%", gross_weight_error, 0.0, 1e-12, "Gross deflator input weights."),
    ("Net nominal shares sum to 100%", net_weight_error, 0.0, 1e-12, "Net deflator input weights."),
    ("Gross chained log changes reproduce", central_real["gross_dlog_error"], 0.0, 1e-12, "Calculated deflator versus weighted dlog inputs."),
    ("Net chained log changes reproduce", central_real["net_dlog_error"], 0.0, 1e-12, "Calculated deflator versus weighted dlog inputs."),
    ("Actual-quarter component prices complete", float(price_missing_count), 0.0, 0.0, "All five price components present."),
    ("GDP counterfactual levels remain positive", max(0.0, 1.0 - counterfactual_minimum), 0.0, 0.0, "Minimum counterfactual GDP must exceed one billion chained dollars."),
    ("Recognition scenarios conserve cumulative gross dollars", scenario_conservation_error, 0.0, 1e-8, "Central, immediate, and delayed timing cases."),
    ("No CAPEX observation is imputed", float(panel["data_status"].astype(str).str.contains("imputed").sum()), 0.0, 0.0, "Missing source values remain missing."),
]

checks = pd.DataFrame(
    check_specs,
    columns=["check", "actual", "expected", "tolerance", "notes"],
)
checks["difference"] = checks["actual"] - checks["expected"]
checks["status"] = np.where(checks["difference"].abs() <= checks["tolerance"], "OK", "REVIEW")
checks = checks[["check", "actual", "expected", "difference", "tolerance", "status", "notes"]]

latest_real = real_contributions.loc[REAL_CONTRIBUTION_END]
summary_output = pd.DataFrame(
    [
        ("Model", "Central methodology", "Nominal gross -> nominal imports -> nominal net -> separate chained deflators", "Real net is never calculated by subtracting real imports."),
        ("Model", "Latest actual quarter", str(REAL_CONTRIBUTION_END), "Real contribution endpoint."),
        ("Imports", "Central IT import share", CENTRAL_IMPORT_SHARES["it_equipment"], "Linked to Import Diagnostics in the workbook."),
        ("Imports", "Central power/cooling import share", CENTRAL_IMPORT_SHARES["power_cooling_equipment"], "Literal C276RC / LA001149 direction."),
        ("Output", "2026 nominal net AI GDP flow ($bn)", float(annual_results.loc[2026, "net_ai_gdp_flow_bn"]), "Includes actual through Q2 and reported estimates thereafter."),
        ("Output", f"{REAL_CONTRIBUTION_END} net AI contribution (pp)", float(latest_real["real_net_ai_contribution_pp"]), "Approximate q/q annualized contribution."),
        ("Checks", "Checks requiring review", int(checks["status"].eq("REVIEW").sum()), "Linked COUNTIF formula in the workbook."),
        ("Caveat", "Attribution perimeter", "Upper-bound central case", "All non-baseline-hyperscaler reported CAPEX is treated as AI-related."),
        ("Caveat", "Real additivity", "Approximate", "Custom chained-dollar flows are not official additive BEA components."),
    ],
    columns=["section", "metric", "value", "notes"],
)

display(checks)
if checks["status"].eq("REVIEW").any():
    warnings.warn("One or more v3 reconciliation checks require review.")


                                                      check  actual  expected  difference  tolerance status                                                                       notes
0                               Category splits sum to 100%  0.0000    0.0000      0.0000     0.0000     OK                                        Every profile fully allocates CAPEX.
1                                   Lag weights sum to 100%  0.0000    0.0000      0.0000     0.0000     OK                                    Recognition conserves each asset dollar.
2        Matched hyperscaler 2022 quarters have zero excess  0.0000    0.0000      0.0000     0.0000     OK                                             Same-quarter baseline identity.
3         Non-hyperscaler attribution equals reported CAPEX  0.0000    0.0000      0.0000     0.0000     OK                                        Central literal-framework perimeter.
4                           US source allocation reconciles  0.0000    0.0000   

In [7]:
def json_value(value):
    if value is None:
        return None
    if isinstance(value, pd.Period):
        return str(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        value = value.item()
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(value, (str, int, float, bool)):
        return value
    return str(value)


def prepare_frame(frame):
    output = frame.copy()
    for column in output.columns:
        if isinstance(output[column].dtype, pd.PeriodDtype):
            output[column] = output[column].astype(str)
        else:
            output[column] = output[column].map(json_value)
    return output


def sheet_payload(name, frame, note):
    clean = prepare_frame(frame)
    return {
        "name": name,
        "title": f"AI CAPEX-GDP Simplified v3 - {name}",
        "note": note,
        "columns": list(clean.columns),
        "rows": [[json_value(value) for value in row] for row in clean.itertuples(index=False, name=None)],
    }


sheets = [
    sheet_payload("Summary", summary_output, "Central results, linked outputs, and the most important interpretation warnings."),
    sheet_payload("Annual Net Flow", annual_output, "Annual nominal company CAPEX, US AI investment, imports, and net AI GDP flow; $ billions."),
    sheet_payload("Quarterly Net Flow", quarterly_output, "Quarterly nominal source and recognized flows; $ billions, not SAAR."),
    sheet_payload("Real GDP Contribution", real_output, "Gross and net flows are deflated independently; implied imports are a contribution difference, not additive real dollars."),
    sheet_payload("Company Detail", company_output, "Central company-level CAPEX attribution and US-share calculation. Missing source observations remain missing."),
    sheet_payload("Industry Breakdown", industry_breakdown, "Central source-quarter attribution by GICS industry group; $ billions."),
    sheet_payload("Asset Breakdown", asset_breakdown, "Recognized nominal gross, imports, and net flow by asset and year; $ billions."),
    sheet_payload("Coverage Audit", coverage_output, "Reported-company coverage and the central broad attribution perimeter by quarter."),
    sheet_payload("Baseline Audit", baseline_audit, "Availability of same-quarter 2022 observations; mandatory only for the five baseline hyperscalers."),
    sheet_payload("Assumptions", assumptions_output, "User-editable model assumptions echoed from the notebook. Blue values denote model inputs."),
    sheet_payload("BEA Series", series_audit, "Official BEA series codes, validated descriptions, data spans, units, and source URLs."),
    sheet_payload("Checks", checks, "One assertion per row. All checks must be OK or explicitly reviewed."),
    sheet_payload("Import Diagnostics", import_diagnostics, "Latest two complete BEA quarters; semiconductors are excluded from the IT numerator."),
    sheet_payload("Gross Deflator", gross_deflator_output, "Törnqvist-style gross AI investment price index and independently deflated real gross flow."),
    sheet_payload("Net Deflator", net_deflator_output, "Törnqvist-style price index built after nominal import subtraction; primary real net series."),
    sheet_payload("Deflator Weights", deflator_weights_output, "Gross and net nominal shares, two-period weights, and underlying component price indexes."),
    sheet_payload("Scenario Comparison", scenario_comparison, "One-at-a-time alternatives; every scenario reruns the full nominal and real calculation."),
]

summary_links = [
    {
        "type": "link",
        "summaryKeyColumn": "metric",
        "summaryKeyValue": "Central IT import share",
        "summaryValueColumn": "value",
        "targetSheet": "Import Diagnostics",
        "targetKeyColumn": "quarter",
        "targetKeyValue": "Two-quarter average",
        "targetValueColumn": "it_import_ratio_clipped",
    },
    {
        "type": "link",
        "summaryKeyColumn": "metric",
        "summaryKeyValue": "Central power/cooling import share",
        "summaryValueColumn": "value",
        "targetSheet": "Import Diagnostics",
        "targetKeyColumn": "quarter",
        "targetKeyValue": "Two-quarter average",
        "targetValueColumn": "power_import_ratio_literal_clipped",
    },
    {
        "type": "link",
        "summaryKeyColumn": "metric",
        "summaryKeyValue": "2026 nominal net AI GDP flow ($bn)",
        "summaryValueColumn": "value",
        "targetSheet": "Annual Net Flow",
        "targetKeyColumn": "year",
        "targetKeyValue": 2026,
        "targetValueColumn": "net_ai_gdp_flow_bn",
    },
    {
        "type": "link",
        "summaryKeyColumn": "metric",
        "summaryKeyValue": f"{REAL_CONTRIBUTION_END} net AI contribution (pp)",
        "summaryValueColumn": "value",
        "targetSheet": "Real GDP Contribution",
        "targetKeyColumn": "quarter",
        "targetKeyValue": str(REAL_CONTRIBUTION_END),
        "targetValueColumn": "real_net_ai_contribution_pp",
    },
    {
        "type": "countif",
        "summaryKeyColumn": "metric",
        "summaryKeyValue": "Checks requiring review",
        "summaryValueColumn": "value",
        "targetSheet": "Checks",
        "targetValueColumn": "status",
        "criteria": "REVIEW",
    },
]

payload = {
    "model": "AI CAPEX–GDP Simplified v3",
    "generated_from": str(ROOT / "ai_capex_gdp_simplified_v3.ipynb"),
    "sheets": sheets,
    "summaryLinks": summary_links,
}

if EXPORT_OUTPUTS:
    if not BUNDLED_NODE.exists():
        raise FileNotFoundError(f"Bundled Node.js runtime not found: {BUNDLED_NODE}")
    if not WORKBOOK_BUILDER.exists():
        raise FileNotFoundError(f"Workbook builder not found: {WORKBOOK_BUILDER}")
    payload_path = OUTPUT_DIR / ".v3_workbook_payload.json"
    workbook_path = OUTPUT_DIR / OUTPUT_FILE
    payload_path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    completed = subprocess.run(
        [
            str(BUNDLED_NODE),
            str(WORKBOOK_BUILDER),
            "--payload",
            str(payload_path),
            "--output",
            str(workbook_path),
        ],
        cwd=ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    payload_path.unlink(missing_ok=True)
    print(f"Saved {workbook_path.relative_to(ROOT)}")


V3_SUMMARY_INSPECTION
{"kind":"table","sheet":"Summary","address":"A1:D25","rows":25,"cols":4,"values":[["AI CAPEX-GDP Simplified v3 - Summary","AI CAPEX-GDP Simplified v3 - Summary","AI CAPEX-GDP Simplified v3 - Summary","AI CAPEX-GDP Simplified v3 - Summary"],["Central results, linked outputs, and the most important interpretation warnings.","Central results, linked outputs, and the most important interpretation warnings.","Central results, linked outputs, and the most important interpretation warnings.","Central results, linked outputs, and the most important interpretation warnings."],[null,null,null,null],["section","metric","value","notes"],["Model","Central methodology","Nominal gross -> nominal imports -> nominal net -> separate chained deflators","Real net is never calculated by subtracting real imports."],["Model","Latest actual quarter","2026Q2","Real contribution endpoint."],["Imports","Central IT import share",0.9373723332192145,"Linked to Import Diagnostics in the workboo

## 7. Interpretation caveats

- The central perimeter is deliberately broad: all non-baseline-hyperscaler CAPEX is treated as AI-related and should be read as an upper-bound attribution assumption.
- The hyperscaler baseline estimates incremental investment relative to 2022, not total AI investment.
- The IT import ratio excludes semiconductor imports but still compares import categories with a narrower private-investment denominator.
- The literal power/cooling formula uses investment divided by imports; its conventional inverse is reported only as a scenario.
- Latest-two-quarter import assumptions applied historically introduce look-ahead and vintage sensitivity.
- Company cash CAPEX timing is not identical to BEA investment recognition; the recognition weights are explicit modeling assumptions.
- Netting occurs in nominal dollars before deflation. Gross and net receive separate chained indexes, and no additive real-import identity is asserted.
- Custom chained-dollar flows are not strictly additive to official real GDP, so the contribution estimates are approximate rather than official BEA statistics.
